[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/neutrino_isospin/neutrino_isospin.ipynb)

# Neutrino Flavor Oscillations

This example computes how neutrinos change flavor as they travel through vacuum, across astrophysical baselines, and out of the solar core. Neutrinos are produced and detected in flavor states yet propagate as mass states with distinct phase rates, so an electron neutrino oscillates into a muon neutrino and back; over long baselines the mass wavepackets separate into an incoherent mixture, while inside the Sun a slowly falling electron density tilts the Hamiltonian gradually enough that an outgoing neutrino follows the local eigenstate and emerges mostly as a muon neutrino.

A two-flavor density state is modelled as `0.5 * (antiscalar + isospin)`, placing pure flavors on the unit flavor sphere with the electron flavor at the intrinsic turning-plane patch `phase_plane.dual()`. Pairing with the phase plane `phase_plane & isospin` reads the electron-flavor projection directly, coherent propagation under an extrinsic Hamiltonian bivector acts via `-Bivector.commutator(State)`, matter shifts that bivector along `phase_plane`, and wavepacket separation dephases the state by a Lindblad half-turn `(mass_plane >> State) - back.anticommutator(State)` that preserves the antiscalar trace.

In $2\times 2$ density-matrix and Pauli-vector notation, a two-flavor state is expanded as $\rho = \tfrac12(I_2 + \mathbf P\cdot\boldsymbol\sigma)$ on complex Hermitian matrices, coherent and matter-shifted evolution obeys $\partial_x\rho = -i[H,\rho]$ with $H = \frac{\Delta m^2}{4E}\begin{pmatrix}-\cos 2\theta & \sin 2\theta\\ \sin 2\theta & \cos 2\theta\end{pmatrix} + \begin{pmatrix}\sqrt{2}\,G_F N_e(x) & 0\\ 0 & 0\end{pmatrix}$, and wavepacket decoherence adds the Lindblad dissipator $\gamma\,(L\rho L^\dagger - \tfrac12\{L^\dagger L,\rho\})$ for the rotated mass-basis projector $L = U\sigma_z U^\dagger$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import show_states
from examples.quantum.neutrino_isospin import render

np.set_printoptions(precision=3, suppress=True)

# Geometric algebra of three-dimensional Euclidean space (+ extrinsic, - intrinsic).
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Bivector = ga.gatype.bivector()                                              # Bivector+
Isospin = ga.gatype.antibivector()                                           # Antibivector-
State = ga.gatype(Isospin + ga.gatype.antiscalar())                          # Antibivector- + Antiscalar-
Rates = ga.gatype((State, State))                                            # State- <- State-
Evolution = ga.gatype((State, State))                                        # State- <- State-
Generator = ga.gatype((State, Bivector, State))                              # State- <- (Bivector+, State-)

one = mv.scalar([1])                                                         # [] Scalar+
antiscalar = one.dual()                                                      # [] Antiscalar-
# Phase plane of a pure flavor state and the plane in which vacuum mixing tilts it.
phase_plane = mv.xy                                                          # [] Bivector+
mixing_plane = mv.xz                                                         # [] Bivector+
# Commutation with a bivector has no antiscalar component; casting embeds its output in the full state space.
coherent_generator = (-Bivector.commutator(State)).cast(Generator)           # [] State- <- (Bivector+, State-)
# Pure electron and muon flavors sit at opposite poles as patches of the phase plane.
flavor_e = phase_plane.dual()                                                # [] Isospin-
flavor_mu = -flavor_e                                                        # [] Isospin-


def density_state(isospin: Isospin) -> State:
    """Adding `antiscalar` to `isospin` (`[...] Isospin-`) and halving forms the trace-one density state
    (`[...] State-`)."""
    return 0.5 * (antiscalar + isospin)                                      # [...] State-


def flavor_probabilities(isospin: Isospin) -> tuple[Scalar, Scalar]:
    """Pairing `isospin` (`[...] Isospin-`) with `phase_plane` gives the electron and muon flavor probabilities
    (`[...] Scalar+`)."""
    electron_part = phase_plane & isospin                                    # [...] Scalar+
    return 0.5 * (one + electron_part), 0.5 * (one - electron_part)

## Coherent oscillation in vacuum

In vacuum a neutrino state precesses at a steady rate in a fixed plane formed by tilting `phase_plane` through the mixing angle in `mixing_plane`. Because the spatial step is uniform and the Hamiltonian is constant, exponentiating the rate map once builds a single `State- <- State-` transfer map that advances the density state across every step.

In [ ]:
omega = 1.0
theta = np.radians(30.0)
n_steps = 300
n_frames = 63
frame_duration_ms = 40

In [ ]:
def vacuum_hamiltonian(omega: float, theta: float) -> Bivector:
    """Tilting `-phase_plane` by mixing angle `theta` in `mixing_plane` gives the vacuum Hamiltonian bivector at
    rate `omega` (`[] Bivector+`)."""
    mixing = (mixing_plane * (-theta)).exp()                                 # [] Rotor+
    return omega * (mixing >> (-phase_plane))                                # [] Bivector+


def step_evolution(rates: Rates, step_size: float) -> Evolution:
    """Truncating the exponential series of `rates * step_size` at fifth order gives the single-step transfer map
    (`[...] State- <- State-`)."""
    small = rates * step_size                                                # [...] State- <- State-
    term = total = State
    for order in range(1, 6):
        term = small(term) / order                                           # [...] State- <- State-
        total = total + term
    return total                                                             # [...] State- <- State-

In [ ]:
def channel_isospins(initial_state: State, transfer: Evolution, n_steps: int) -> Iterator[Isospin]:
    """Yields the state's isospin (`[] Isospin-`) across `n_steps` applications of `transfer`."""
    density = initial_state                                                  # [] State-
    for _ in range(n_steps):
        yield 2 * density.select[1]                                          # [] Isospin-
        density = transfer(density)                                          # [] State-


vacuum = vacuum_hamiltonian(omega, theta)                                    # [] Bivector+
wavelength = 4.0 * np.pi / omega
distances, step_size = np.linspace(0.0, 2.0 * wavelength, n_steps, retstep=True)
turning = coherent_generator(vacuum)                                         # [] State- <- State-
transfer = step_evolution(turning, step_size)                                # [] State- <- State-
vacuum_history = show_states(
    channel_isospins(density_state(flavor_e), transfer, n_steps),
    render.precession_animation, vacuum, distances, n_frames,
    name="neutrino_vacuum_precession", duration_ms=frame_duration_ms, n_frames=n_frames,
);

On the flavor sphere at left, the indigo trajectory traces a cone around the dark Hamiltonian great circle from the green start marker at the blue electron-flavor pole back to itself, while on the right the solid blue electron survival curve and dashed red muon appearance curve trade probability periodically.

## Wavepacket decoherence over long baselines

Because the two mass eigenstates propagate at slightly different group velocities, their wavepackets gradually lose overlap over astrophysical baselines. Sandwiching the state in the mass bivector `mass_plane >> State` reflects the isospin components transverse to the mass axis, so subtracting the trace-balancing anticommutator damps coherence between mass states while keeping total probability equal to one.

In [ ]:
gamma = 0.05
decoherence_distance = 100.0
n_decoherence_steps = 400

In [ ]:
def relaxation(process: Bivector) -> Rates:
    """Sandwiching a state in `process` and subtracting half the squared reverse norm on each side gives a
    trace-preserving Lindblad rate map (`[] State- <- State-`)."""
    back = process.reverse().symmetric_reverse_product()                     # [] Scalar+
    return (process >> State) - back.anticommutator(State)                   # [] State- <- State-


mixing = (mixing_plane * (-theta)).exp()                                     # [] Rotor+
mass_plane = mixing >> phase_plane                                           # [] Bivector+
dephasing = gamma * relaxation(mass_plane)                                   # [] State- <- State-
decoherence_distances, decoherence_step = np.linspace(0.0, decoherence_distance, n_decoherence_steps, retstep=True)
decoherence_transfer = step_evolution(turning + dephasing, decoherence_step) # [] State- <- State-
asymptotic_probability = float(1.0 - 0.5 * np.sin(2.0 * theta) ** 2)
decoherence_history = show_states(
    channel_isospins(density_state(flavor_e), decoherence_transfer, n_decoherence_steps),
    render.decoherence_animation, vacuum, decoherence_distances, asymptotic_probability, n_frames,
    name="neutrino_decoherence", duration_ms=frame_duration_ms, n_frames=n_frames,
);

The isospin spirals inward along the precession cone onto the mass-plane axis inside the unit ball, and the oscillating blue and red flavor probabilities damp onto the grey dashed incoherent average.

## Adiabatic flavor conversion in the Sun

Coherent forward scattering on solar electrons adds a matter potential along `phase_plane` that starts well above the Mikheyev-Smirnov-Wolfenstein resonance in the core and decays exponentially toward the surface. Because the Hamiltonian plane tilts slowly relative to the rapid precession rate, an electron neutrino born aligned with the high-density plane stays locked to the instantaneous plane across resonance and exits near the muon pole.

In [ ]:
solar_theta = np.radians(10.0)
solar_distance = 100.0
n_solar_steps = 600
core_over_resonance = 3.0
density_scale = 25.0

In [ ]:
def adiabatic_isospins(initial_state: State, rates: Rates, planes: Bivector, step_size: float
                       ) -> Iterator[tuple[Isospin, Bivector]]:
    """Yields the local isospin and Hamiltonian plane (`[] Isospin-`, `[] Bivector+`) as `rates` (`[n_steps]`)
    varies along the trajectory."""
    n_local_steps = rates.shape[0]
    density = initial_state                                                  # [] State-
    for step in range(n_local_steps):
        yield 2 * density.select[1], planes[step]                            # [] Isospin-, [] Bivector+
        density = step_evolution(rates[step], step_size)(density)            # [] State-


solar_vacuum = vacuum_hamiltonian(omega, solar_theta)                        # [] Bivector+
resonance = omega * np.cos(2.0 * solar_theta)
solar_distances, solar_step = np.linspace(0.0, solar_distance, n_solar_steps, retstep=True)
potentials = core_over_resonance * resonance * np.exp(-solar_distances / density_scale)
planes = solar_vacuum + potentials * phase_plane                             # [n_solar_steps] Bivector+
solar_rates = coherent_generator(planes)                                     # [n_solar_steps] State- <- State-
solar_history = show_states(
    adiabatic_isospins(density_state(flavor_e), solar_rates, planes, solar_step),
    render.adiabatic_animation, solar_distances, potentials, planes[0], planes[-1], n_frames,
    name="neutrino_solar_msw", duration_ms=frame_duration_ms, n_frames=n_frames,
);

As the dotted grey matter potential falls away, the solid dark Hamiltonian great circle tilts away from the dotted initial circle near the equator and carries the tightly spiralling indigo isospin path from the blue electron pole down to the red muon pole.

In [ ]:
# checks
pole_e, pole_mu = flavor_probabilities(flavor_e)
assert ((pole_e - 1).abs() < 1e-12).all() and (pole_mu.abs() < 1e-12).all()

vacuum_path = stack(vacuum_history)                                          # [n_steps] Isospin-
vacuum_electron, vacuum_muon = flavor_probabilities(vacuum_path)             # [n_steps] Scalar+, [n_steps] Scalar+
expected_vacuum = 1.0 - np.sin(2.0 * theta) ** 2 * np.sin(0.5 * omega * distances) ** 2
assert ((vacuum_electron + vacuum_muon - 1).abs() < 1e-10).all()
assert ((vacuum_electron - expected_vacuum).abs() < 1e-4).all()

decoherence_path = stack(decoherence_history)                                # [n_decoherence_steps] Isospin-
decoherence_electron, _ = flavor_probabilities(decoherence_path)             # [n_decoherence_steps] Scalar+
assert ((decoherence_electron[-1] - asymptotic_probability).abs() < 2e-3).all()
# Trace conservation under combined coherent precession and Lindblad dephasing.
dephased_state = decoherence_transfer(density_state(flavor_e))            # [] State-
assert (((one & dephased_state) - 0.5).abs() < 1e-12).all()

solar_path = stack([spin for spin, _ in solar_history])                      # [n_solar_steps] Isospin-
solar_electron, _ = flavor_probabilities(solar_path)                         # [n_solar_steps] Scalar+
assert (solar_electron[0] > 0.99).all() and (solar_electron[-1] < 0.05).all()